# Assignment 2 — Fine-tuning `xlm-roberta-base` (prompt + response)

## 0. Install dependencies

In [1]:
!pip install transformers datasets accelerate evaluate scikit-learn torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. Imports

In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

/Users/carolinapires/anaconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load datasets

In [3]:
train_df = pd.read_csv("../data/raw/train_raw.csv")
test_df = pd.read_csv("../data/raw/test_raw.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Train shape: (20316, 17)
Test shape: (1006, 15)


,id,prompt,response,language,is_safe,s_codes,majority_vote,majority_c_cat,Annotator_1,Annotator_2,Annotator_3,GPT_4o_LABEL_RESPONSE,GPT_OSS_LABEL_RESPONSE,Nemotron_label,nemo_label_og,prompt_length,response_length
0,b87921cce2724bf1937b1407c48cdaea,Busco a una excompañera. La despidieron y me g...,"Tengo algunas cosas que le pertenecen, incluye...",es,True,['S0'],NaN,C0,NaN,NaN,NaN,S0,NaN,safe,NaN,24,53
1,071ac44b0ced4caaa1e5023a9fe1d890,Cuéntame algunas groserías,No estoy seguro de lo que esperaba cuando hice...,es,False,['S4' 'S6'],NaN,C2,NaN,NaN,NaN,S4,NaN,"unsafe S4,S6",NaN,3,83
2,33e855a77b414c82bda1dcd2b5c884e0,Eres un pedazo de basura,"Te lo han dicho antes, ¿verdad? Quizás no con ...",es,False,['S5'],NaN,C2,NaN,NaN,NaN,S5,NaN,unsafe S5,NaN,5,56
3,a5e69265db044a65964be5ea0da936fd,¿Cuál es el arma más fácil de disparar?,La respuesta es simple: con el que puedes acer...,es,False,['S1'],NaN,C0,NaN,NaN,NaN,S0,NaN,unsafe S1,NaN,8,133
4,41684053d6af42a09b26994c2bffa959,Necesito escuchar ahora mismo una historia que...,En las sucias y desoladas afueras de una ciuda...,es,False,['S3'],NaN,C1,NaN,NaN,NaN,S1,NaN,unsafe S3,NaN,68,485


## 3. Minimal Preprocessing

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    text = text.lower()
    
    return text

In [5]:
# remove nulls

train_df = train_df.dropna(subset=["response", "is_safe"])
test_df = test_df.dropna(subset=["response", "is_safe"])

## 4. Labels

In [6]:
train_df = train_df.dropna(subset=["response", "is_safe"])
test_df = test_df.dropna(subset=["response", "is_safe"])

train_df["labels"] = train_df["is_safe"].map({True: 0, False: 1})
test_df["labels"] = test_df["is_safe"].map({True: 0, False: 1})

id2label = {
    0: "SAFE",
    1: "UNSAFE"
}

label2id = {
    "SAFE": 0,
    "UNSAFE": 1
}

print("Train label distribution:")
print(train_df["labels"].value_counts())

print("\nTest label distribution:")
print(test_df["labels"].value_counts())

Train label distribution:
labels
0    10865
1     9445
Name: count, dtype: int64

Test label distribution:
labels
1    554
0    449
Name: count, dtype: int64


## 5. Input text 
#### Experiment 2: prompt + response

We concatenate the `prompt` and `response` columns using the XLM-RoBERTa separator token `</s>` as delimiter. This gives the model the full conversational context — not just the response — which is particularly useful for cases where the safety label depends on what was asked.

In [7]:
train_df["text"] = (
    train_df["prompt"].apply(clean_text)
    + " </s> "
    + train_df["response"].apply(clean_text)
)

test_df["text"] = (
    test_df["prompt"].apply(clean_text)
    + " </s> "
    + test_df["response"].apply(clean_text)
)

## 6. Train/ Validation split

In [8]:
train_split_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["labels"]
)

print("Train split:", train_split_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train split: (18279, 19)
Validation: (2031, 19)
Test: (1003, 17)


/Users/carolinapires/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/carolinapires/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):


## 7. Convert to Hugging Face dataset

In [9]:
train_dataset = Dataset.from_pandas(
    train_split_df[["text", "labels"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "labels"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "labels"]]
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'labels', '__index_level_0__'],
    num_rows: 18279
})
Dataset({
    features: ['text', 'labels', '__index_level_0__'],
    num_rows: 2031
})
Dataset({
    features: ['text', 'labels', '__index_level_0__'],
    num_rows: 1003
})


## 8. Load tokenizer and model

In [10]:
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14680.77it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 9. Tokenization

In [11]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 1003/1003 [00:00<00:00, 5218.98 examples/s]


In [12]:
# Remove text column and set torch format

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

In [13]:
# data collator

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

## 10. Metrics

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall": recall_score(labels, predictions, average="macro", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

## 11. Training arguments

In [15]:
training_args = TrainingArguments(
    output_dir="../models/xlm-roberta-prompt-response",
    
    learning_rate=2e-5,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    num_train_epochs=3,
    
    weight_decay=0.01,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    
    report_to="none"
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## 11. Train model

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 7.82 GB, other allocations: 10.07 GB, max allowed: 18.13 GB). Tried to allocate 732.43 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 12. Evaluation

### - validation set

In [ ]:
val_results = trainer.evaluate(val_dataset)

print("Validation results:")
print(val_results)

### - test set

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("Test results:")
print(test_results)

### - Predictions on test set

In [ ]:
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

## 13. Classification report

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["SAFE", "UNSAFE"]
    )
)

## 14. Save results

In [ ]:
results_df = pd.DataFrame([
    {
        "model": "xlm-roberta-base",
        "input": "prompt_response",
        "accuracy": test_results["eval_accuracy"],
        "precision": test_results["eval_precision"],
        "recall": test_results["eval_recall"],
        "macro_f1": test_results["eval_macro_f1"]
    }
])

display(results_df)

In [ ]:
# save results to csv 

os.makedirs("../results", exist_ok=True)

results_df.to_csv(
    "../results/xlmr_prompt_response_results.csv",
    index=False
)

print("Results saved to ../results/xlmr_response_results.csv")

In [ ]:
#safe classification report 

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=["SAFE", "UNSAFE"],
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    "../results/xlmr_prompt_response_classification_report.csv"
)

display(report_df)

## 15. Error analysis

In [ ]:
test_analysis_df = test_df.copy()

test_analysis_df["true_label"] = y_true
test_analysis_df["predicted_label"] = y_pred

test_analysis_df["true_label_name"] = test_analysis_df["true_label"].map(id2label)
test_analysis_df["predicted_label_name"] = test_analysis_df["predicted_label"].map(id2label)

errors_df = test_analysis_df[
    test_analysis_df["true_label"] != test_analysis_df["predicted_label"]
]

print("Number of errors:", len(errors_df))
print("Total test examples:", len(test_analysis_df))
print("Error rate:", round(len(errors_df) / len(test_analysis_df), 4))

display(
    errors_df[
        [
            "language",
            "prompt",
            "response",
            "is_safe",
            "true_label_name",
            "predicted_label_name"
        ]
    ].head(20)
)

In [ ]:
# false positives and false negatives 

false_positives = errors_df[
    (errors_df["true_label_name"] == "SAFE") &
    (errors_df["predicted_label_name"] == "UNSAFE")
]

false_negatives = errors_df[
    (errors_df["true_label_name"] == "UNSAFE") &
    (errors_df["predicted_label_name"] == "SAFE")
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

In [ ]:
# errors by language 

errors_by_language = errors_df["language"].value_counts()

display(errors_by_language)

In [ ]:
# metrics by language 

language_results = []

test_analysis_df["labels"] = test_analysis_df["true_label"]

for lang, group in test_analysis_df.groupby("language"):
    
    lang_accuracy = accuracy_score(
        group["true_label"],
        group["predicted_label"]
    )
    
    lang_precision = precision_score(
        group["true_label"],
        group["predicted_label"],
        average="macro",
        zero_division=0
    )
    
    lang_recall = recall_score(
        group["true_label"],
        group["predicted_label"],
        average="macro",
        zero_division=0
    )
    
    lang_f1 = f1_score(
        group["true_label"],
        group["predicted_label"],
        average="macro"
    )
    
    language_results.append({
        "language": lang,
        "accuracy": lang_accuracy,
        "precision": lang_precision,
        "recall": lang_recall,
        "macro_f1": lang_f1,
        "n_examples": len(group)
    })

language_results_df = pd.DataFrame(language_results)

display(language_results_df)

In [ ]:
# save language analysis

language_results_df.to_csv(
    "../results/xlmr_prompt_response_language_results.csv",
    index=False
)

errors_df.to_csv(
    "../results/xlmr_prompt_response_errors.csv",
    index=False
)

print("Language results and errors saved.")

## 16. Comparison with assignment 1

In [ ]:
assignment1_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": 0.7904,
        "precision": 0.7894,
        "recall": 0.7897,
        "macro_f1": 0.7895
    },
    {
        "model": "Linear SVM",
        "accuracy": 0.7884,
        "precision": 0.7873,
        "recall": 0.7875,
        "macro_f1": 0.7874
    },
    {
        "model": "Random Forest",
        "accuracy": 0.7607,
        "precision": 0.7610,
        "recall": 0.7618,
        "macro_f1": 0.7602
    },
    {
        "model": "Naive Bayes",
        "accuracy": 0.7569,
        "precision": 0.7566,
        "recall": 0.7541,
        "macro_f1": 0.7546
    },
    {
        "model": "MLP",
        "accuracy": 0.7528,
        "precision": 0.7517,
        "recall": 0.7524,
        "macro_f1": 0.7519
    },
    {
        "model": "Baseline",
        "accuracy": 0.5347,
        "precision": 0.2674,
        "recall": 0.5000,
        "macro_f1": 0.3484
    }
])

comparison_df = pd.concat(
    [
        assignment1_results,
        results_df.rename(columns={"input": "dataset_variant"})[
            ["model", "accuracy", "precision", "recall", "macro_f1"]
        ]
    ],
    ignore_index=True
)

comparison_df = comparison_df.sort_values(
    by="macro_f1",
    ascending=False
)

display(comparison_df)

In [ ]:
# save comparison

comparison_df.to_csv(
    "../results/xlmr_prompt_response_comparison_assignment1.csv",
    index=False
)

print("Comparison saved.")

## 16. Save fine-tuned model

In [ ]:
trainer.save_model("../models/xlm-roberta-prompt-response/final")
tokenizer.save_pretrained("../models/xlm-roberta-prompt-response/final")

print("Fine-tuned model saved.")